In [21]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")

In [22]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool
def get_current_time(timezone: str, location: str) -> str:
    """지정한 지역의 현재 시각을 반환합니다."""
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    return f"{timezone} ({location}) 현재시각 {now}"

tools = [get_current_time]
tool_dict = {"get_current_time": get_current_time}
llm_with_tools = llm.bind_tools(tools)

In [23]:
for c in llm.stream([HumanMessage("잘 지냈어? 한국 사회의 문제점에 대해 이야기해줘.")]):
    print(c.content, end='|') 

|저|는| 잘| 지|내|고| 있습니다|!| 한국| 사회|의| 문제|점|에| 대해서| 이야기|해|볼|게|요|.| 여러| 가지| 이|슈|가| 있지만|,| 특히| 두| 가지| 주요| 문제|를| 언|급|할| 수| 있습니다|.

|1|.| **|고|용| 문제|**|:| 한국|은| 청|년| 실|업|률|이| 높은| 편|이며|,| 일|자리|의| 질|도| 문제가| 됩니다|.| 많은| 청|년|들이| 안정|적인| 일|자|리를| 찾|기| 어려|워|하며|,| 비|정|규|직| 일|자|리가| 늘|어나|고| 있습니다|.| 이러한| 상황|은| 젊|은| 세|대|의| 경제|적| 불|안|정을| 초|래|하고| 있으며|,| 사회|적| 불|만|을| 증가|시키|고| 있습니다|.

|2|.| **|경제|적인| 불|평|등|**|:| 소|득| 불|평|등|과| 부|의| 집중| 문제|도| 심|각|합니다|.| 대|기업|과| 중|소|기업|,| 그리고| 개인| 간|의| 경제|적| 격|차|가| 커|지고| 있으며|,| 이는| 사회|적| 통|합|을| 어렵|게| 하고| 있습니다|.| 부|동|산| 가격| 상승|으로| 인해| 주|택| 구|입|이| 어려|워|지고|,| 젊|은| 세|대|의| 경제|적| 부담|이| 가|중|되고| 있습니다|.

|이| 외|에도| 양|극|화|,| 교육| 경쟁|,| 고|령|화| 사회| 등| 다양한| 사회|적| 이|슈|가| 있습니다|.| 이러한| 문제|들을| 해결|하기| 위해|서는| 정부|의| 정책|뿐|만| 아니라|,| 사회| 전|반|의| 변화|와| 협|력이| 필요|합니다|.||||

In [24]:
from langchain_core.messages import SystemMessage

messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

response = llm_with_tools.stream(messages)

# 파편화된 tool_call 청크를 하나로 합치기 
is_first = True
for chunk in response:    
    print("chunk type: ", type(chunk))
    
    if is_first:
        is_first = False
        gathered = chunk
    else:
        gathered += chunk
    
    print("content: ", gathered.content, "tool_call_chunk", gathered.tool_calls)

messages.append(gathered)

chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_00unfN4s61NQ2HNodV2qaMJW', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_00unfN4s61NQ2HNodV2qaMJW', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_00unfN4s61NQ2HNodV2qaMJW', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': ''}, 'id': 'call_00unfN4s61NQ2HNodV2qaMJW', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': 'Asia'}, 'id': 'call_00unfN4s61NQ2HNodV2qaMJW', 'type': 'tool_c

In [25]:
from rich.pretty import pprint
pprint(gathered)

AIMessageChunk(
│   content='',
│   additional_kwargs={},
│   response_metadata={
│   │   'model_provider': 'openai',
│   │   'finish_reason': 'tool_calls',
│   │   'model_name': 'gpt-4o-mini-2024-07-18',
│   │   'system_fingerprint': 'fp_4abd59899c',
│   │   'service_tier': 'default'
│   },
│   id='lc_run--01a07420-3aa2-7ba1-b28a-9263fbb5b0ef',
│   tool_calls=[
│   │   {
│   │   │   'name': 'get_current_time',
│   │   │   'args': {'timezone': 'Asia/Seoul', 'location': 'Busan'},
│   │   │   'id': 'call_00unfN4s61NQ2HNodV2qaMJW',
│   │   │   'type': 'tool_call'
│   │   }
│   ],
│   invalid_tool_calls=[],
│   usage_metadata={
│   │   'input_tokens': 78,
│   │   'output_tokens': 23,
│   │   'total_tokens': 101,
│   │   'input_token_details': {'audio': 0, 'cache_read': 0},
│   │   'output_token_details': {'audio': 0, 'reasoning': 0}
│   },
│   tool_call_chunks=[
│   │   {
│   │   │   'name': 'get_current_time',
│   │   │   'args': '{"timezone":"Asia/Seoul","location":"Busan"}',
│   │   │   'id': 'call_00unfN4s61NQ2HNodV2qaMJW',
│   │   │   'index': 0,
│   │   │   'type': 'tool_call_chunk'
│   │   }
│   ],
│   chunk_position='last'
)

In [26]:
for tool_call in gathered.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # tool_dict를 사용하여 도구 이름으로 도구 함수를 선택
    print(tool_call["args"]) # 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

pprint(messages)

{'timezone': 'Asia/Seoul', 'location': 'Busan'}


[
│   SystemMessage(
│   │   content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.',
│   │   additional_kwargs={},
│   │   response_metadata={}
│   ),
│   HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
│   AIMessageChunk(
│   │   content='',
│   │   additional_kwargs={},
│   │   response_metadata={
│   │   │   'model_provider': 'openai',
│   │   │   'finish_reason': 'tool_calls',
│   │   │   'model_name': 'gpt-4o-mini-2024-07-18',
│   │   │   'system_fingerprint': 'fp_4abd59899c',
│   │   │   'service_tier': 'default'
│   │   },
│   │   id='lc_run--01a07420-3aa2-7ba1-b28a-9263fbb5b0ef',
│   │   tool_calls=[
│   │   │   {
│   │   │   │   'name': 'get_current_time',
│   │   │   │   'args': {'timezone': 'Asia/Seoul', 'location': 'Busan'},
│   │   │   │   'id': 'call_00unfN4s61NQ2HNodV2qaMJW',
│   │   │   │   'type': 'tool_call'
│   │   │   }
│   │   ],
│   │   invalid_tool_calls=[],
│   │   usage_metadata={
│   │   │   'input_tokens': 78,
│   │   │   'output_tokens': 23,
│   │   │   'total_tokens': 101,
│   │   │   'input_token_details': {'audio': 0, 'cache_read': 0},
│   │   │   'output_token_details': {'audio': 0, 'reasoning': 0}
│   │   },
│   │   tool_call_chunks=[
│   │   │   {
│   │   │   │   'name': 'get_current_time',
│   │   │   │   'args': '{"timezone":"Asia/Seoul","location":"Busan"}',
│   │   │   │   'id': 'call_00unfN4s61NQ2HNodV2qaMJW',
│   │   │   │   'index': 0,
│   │   │   │   'type': 'tool_call_chunk'
│   │   │   }
│   │   ],
│   │   chunk_position='last'
│   ),
│   ToolMessage(
│   │   content='Asia/Seoul (Busan) 현재시각 2026-09-06 09:31:05',
│   │   name='get_current_time',
│   │   tool_call_id='call_00unfN4s61NQ2HNodV2qaMJW'
│   )
]

In [27]:
for c in llm_with_tools.stream(messages):
    print(c.content, end='|')

|부|산|은| 지금| |202|6|년| |9|월| |6|일| |09|시| |31|분|입니다|.||||